# 02 — Match Rate Semua Predikat

**Tahap 4 ceklis: Audit ulang match rate pasca-normalisasi** · Protokol DSIC-2906-1 bagian 5, 15 (K-04), 18, 20 (Kegiatan 4)

**Pertanyaan:** setelah provinsi tersedia dan penerbit dinormalisasi, bagaimana sebaran **setiap tipe predikat** di bin B1–B5? Protokol bagian 18 mensyaratkan:
- setiap predikat kueri punya **≥ 100 titik eligible** (10 × k, k = 10);
- setiap bin idealnya berisi **lebih dari satu tipe predikat**, agar pengaruh selektivitas tidak tercampur pengaruh tipe predikat.

**Pilihan sementara** (direvisi saat bimbingan):
1. Batas bin mengikuti usulan protokol bagian 15: B1 < 0,1% · B2 0,1–0,5% · B3 0,5–2,5% · B4 2,5–10% · B5 > 10%.
2. Kolom provinsi memakai hasil perapian (menunggu K-01).
3. Berita yang tertaut ke beberapa kabupaten/provinsi/kategori dianggap **cocok jika salah satu isinya cocok**.

## Langkah 1: Membaca korpus dan menyiapkan fungsi pembantu

Fungsi `tabel_match_rate` dipakai ulang untuk setiap tipe predikat: menghitung match rate terhadap **N = 64.354**, menempatkannya ke bin, dan menandai apakah predikat memenuhi syarat ≥ 100 titik (`layak_100`).

In [1]:
# Korpus, jumlah titik, dan batas bin yang dipakai di seluruh notebook
import pandas as pd

corpus = pd.read_parquet("../data/processed/corpus.parquet")
N = len(corpus)

batas = [0, 0.1, 0.5, 2.5, 10, 100]
label = ["B1", "B2", "B3", "B4", "B5"]

# Fungsi pembantu: tabel jumlah titik, match rate, bin, dan kelayakan (>= 100 titik)
def tabel_match_rate(jumlah, tipe):
    t = pd.DataFrame({"jumlah_titik": jumlah})
    t["match_rate_%"] = (t["jumlah_titik"] / N * 100).round(4)
    t["bin"] = pd.cut(t["match_rate_%"], bins=batas, labels=label)
    t["layak_100"] = t["jumlah_titik"] >= 100
    t["tipe"] = tipe
    return t

## Langkah 2: Predikat penerbit (`publisher_id`)

Memakai penerbit hasil normalisasi protokol bagian 17 (3.862 domain → 2.808 penerbit). Predikat penerbit adalah tipe predikat yang berbeda dari wilayah, sehingga berguna untuk mengisi bin B2–B4 dengan lebih dari satu tipe.

In [2]:
# Jumlah berita per penerbit hasil normalisasi (subdomain sudah digabung)
jumlah_penerbit = corpus["publisher_id"].dropna().value_counts()
t_penerbit = tabel_match_rate(jumlah_penerbit, "penerbit")

print("Jumlah penerbit:", len(t_penerbit))
print("Penerbit layak (>= 100 titik):", t_penerbit["layak_100"].sum())
t_penerbit.head(10)

Jumlah penerbit: 2808
Penerbit layak (>= 100 titik): 48


,jumlah_titik,match_rate_%,bin,layak_100,tipe
publisher_id,,,,,
detik.com,4649,7.2241,B4,True,penerbit
antaranews.com,1777,2.7613,B4,True,penerbit
kompas.com,1686,2.6199,B4,True,penerbit
rri.co.id,1401,2.1770,B3,True,penerbit
jawapos.com,1338,2.0791,B3,True,penerbit
polri.go.id,915,1.4218,B3,True,penerbit
disway.id,903,1.4032,B3,True,penerbit
tribunnews.com,852,1.3239,B3,True,penerbit
kompasiana.com,839,1.3037,B3,True,penerbit


## Langkah 3: Predikat kategori

Kategori berbentuk daftar karena berita bisa terkait atraksi berbeda kategori. Akibatnya angka budaya/buatan **lebih besar** dari audit awal (audit awal memakai satu kategori per berita).

In [3]:
# Kategori berbentuk daftar; satu berita bisa terkait beberapa kategori
jumlah_kategori = corpus.explode("category")["category"].value_counts()
t_kategori = tabel_match_rate(jumlah_kategori, "kategori")
t_kategori

,jumlah_titik,match_rate_%,bin,layak_100,tipe
category,,,,,
alam,26238,40.7714,B5,True,kategori
budaya,21823,33.9109,B5,True,kategori
buatan,14474,22.4912,B5,True,kategori
desa,6329,9.8347,B4,True,kategori


## Langkah 4: Predikat temporal

Lima ambang tanggal dari protokol bagian 5.1. Hanya berita yang punya tanggal terbit; dokumen JADESTA tidak memenuhi predikat temporal.

In [4]:
# Jumlah titik yang terbit sejak setiap ambang tanggal (hanya berita yang punya tanggal)
ambang = ["2021-09-01", "2024-01-01", "2025-01-01", "2026-01-01", "2026-07-23"]
jumlah_temporal = pd.Series(
    {f"sejak {a}": (corpus["published_at"] >= a).sum() for a in ambang}
)
t_temporal = tabel_match_rate(jumlah_temporal, "temporal")
t_temporal

,jumlah_titik,match_rate_%,bin,layak_100,tipe
sejak 2021-09-01,42893,66.6516,B5,True,temporal
sejak 2024-01-01,35171,54.6524,B5,True,temporal
sejak 2025-01-01,30472,47.3506,B5,True,temporal
sejak 2026-01-01,22084,34.3164,B5,True,temporal
sejak 2026-07-23,7743,12.0319,B5,True,temporal


## Langkah 5: Predikat majemuk kabupaten ∧ kategori

Setiap pasangan (kabupaten, kategori) yang benar-benar ada di korpus (README bagian 13: predikat majemuk harus dari metadata nyata). `nunique()` mencegah satu dokumen terhitung dua kali setelah dua kali explode.

In [5]:
# Setiap pasangan (kabupaten, kategori); nunique mencegah satu dokumen terhitung dua kali
pasangan = corpus.explode("kabupaten").explode("category")
jumlah_kab_kat = pasangan.groupby(["kabupaten", "category"])["doc_id"].nunique()
jumlah_kab_kat.index = [f"{k} ∧ {c}" for k, c in jumlah_kab_kat.index]
t_kab_kat = tabel_match_rate(jumlah_kab_kat, "kabupaten ∧ kategori")

print("Jumlah pasangan:", len(t_kab_kat), "| layak:", t_kab_kat["layak_100"].sum())

Jumlah pasangan: 1640 | layak: 357


## Langkah 6: Predikat majemuk kabupaten ∧ berita ∧ sejak 2026-01-01

Bentuk predikat majemuk kedua dari protokol bagian 5.2: berita yang terbit sejak 2026 di setiap kabupaten.

In [6]:
# Berita terbit sejak 2026 per kabupaten
berita_2026 = corpus[(corpus["source_type"] == "news") & (corpus["published_at"] >= "2026-01-01")]
jumlah_kab_berita = berita_2026.explode("kabupaten")["kabupaten"].value_counts()
jumlah_kab_berita.index = [f"{k} ∧ news ∧ 2026" for k in jumlah_kab_berita.index]
t_kab_berita = tabel_match_rate(jumlah_kab_berita, "kabupaten ∧ news ∧ 2026")

print("Jumlah kombinasi:", len(t_kab_berita), "| layak:", t_kab_berita["layak_100"].sum())

Jumlah kombinasi: 342 | layak: 134


## Langkah 6b: Predikat tipe sumber (`source_type`)

Predikat wajib untuk kueri Q3 (README bagian 15). Tiga nilai: `news`, `jadesta_atraksi`, `jadesta_desa`. Ketiganya berada di rezim lebar (B4–B5). Ingat temuan T-03: perbedaan recall antar tipe sumber bisa tercampur dengan perbedaan panjang teks (berita hanya judul).

In [8]:
# Predikat tipe sumber: news, jadesta_atraksi, jadesta_desa (wajib untuk kueri Q3)
jumlah_sumber = corpus["source_type"].value_counts()
t_sumber = tabel_match_rate(jumlah_sumber, "tipe sumber")
t_sumber

,jumlah_titik,match_rate_%,bin,layak_100,tipe
source_type,,,,,
news,42893,66.6516,B5,True,tipe sumber
jadesta_atraksi,15132,23.5137,B5,True,tipe sumber
jadesta_desa,6329,9.8347,B4,True,tipe sumber


## Langkah 7: Peta tipe predikat × bin

Hasil utama notebook ini. Tabel hanya menghitung predikat **layak** (≥ 100 titik), karena hanya predikat inilah yang boleh dipakai di query workload.

In [10]:
# Tabel provinsi dan kabupaten dihitung ulang agar semua tipe ada dalam satu peta
t_provinsi = tabel_match_rate(corpus.explode("province")["province"].value_counts(), "provinsi")
t_kabupaten = tabel_match_rate(corpus.explode("kabupaten")["kabupaten"].value_counts(), "kabupaten")

semua = pd.concat([t_provinsi, t_kabupaten, t_penerbit, t_kategori, t_temporal, t_kab_kat, t_kab_berita, t_sumber])
# Jumlah predikat LAYAK (>= 100 titik) per tipe dan bin
layak = semua[semua["layak_100"]]
peta = pd.crosstab(layak["tipe"], layak["bin"])
peta

bin,B2,B3,B4,B5
tipe,,,,
kabupaten,145,93,1,0
kabupaten ∧ kategori,301,56,0,0
kabupaten ∧ news ∧ 2026,111,23,0,0
kategori,0,0,1,3
penerbit,29,16,3,0
provinsi,1,16,16,3
temporal,0,0,0,5
tipe sumber,0,0,1,2


## Ringkasan hasil

Jumlah predikat **layak** (≥ 100 titik) per tipe dan bin:

| Tipe | B1 | B2 | B3 | B4 | B5 |
|---|---:|---:|---:|---:|---:|
| kabupaten | 0 | 145 | 93 | 1 | 0 |
| kabupaten ∧ kategori | 0 | 301 | 56 | 0 | 0 |
| kabupaten ∧ news ∧ 2026 | 0 | 111 | 23 | 0 | 0 |
| penerbit | 0 | 29 | 16 | 3 | 0 |
| provinsi | 0 | 1 | 16 | 16 | 3 |
| kategori | 0 | 0 | 0 | 1 | 3 |
| temporal | 0 | 0 | 0 | 0 | 5 |
| tipe sumber | 0 | 0 | 0 | 1 | 2 |
| **Jumlah tipe** | **0** | **5** | **5** | **5** | **4** |

- **B1 kosong untuk semua tipe.** Syarat ≥ 100 titik sama dengan match rate ≥ 100 / 64.354 = **0,155%**, sedangkan batas atas B1 adalah 0,1%. Jadi B1 **secara matematis tidak mungkin** berisi predikat yang layak.
- **B2 dan B3 kaya**: masing-masing berisi 5 tipe predikat.
- **B4 terisi 5 tipe**, tetapi didominasi provinsi (16 dari 22 predikat). Penerbit menyumbang 3 (detik.com 7,22%, antaranews.com 2,76%, kompas.com 2,62%).
- **B5 berisi 4 tipe**: provinsi, kategori, temporal, tipe sumber.
- Dari 2.808 penerbit, hanya **48** yang layak. Dari 1.640 pasangan kabupaten ∧ kategori, **357** layak.
- Predikat **tipe sumber**: news (66,65%) dan atraksi (23,51%) di B5, desa (9,83%) di B4.

## Kesimpulan dan implikasi (draf)

1. **Ada kontradiksi antara batas bin protokol (K-04) dan syarat kecukupan (bagian 18).** Dengan k = 10 dan syarat 10 × k, predikat paling selektif yang masih layak berada di 0,155%, sehingga bin B1 (< 0,1%) selalu kosong. Pilihan untuk dibahas saat bimbingan:
   - menggeser batas bin, misalnya B1 = 0,155–0,3%;
   - melonggarkan syarat kecukupan khusus rezim paling selektif (misalnya ≥ k = 10 titik, setara 0,0155%), dengan konsekuensi variasi recall lebih kecil;
   - menggabungkan B1 ke B2 dan menyatakan rezim < 0,155% di luar jangkauan korpus.
2. **Rezim selektif (B2–B3) adalah kekuatan korpus**, sejalan dengan catatan penutup protokol: daya uji AdaptiveL paling memadai justru di sini.
3. **Rezim menengah (B4) terisi, tetapi bergantung pada provinsi.** Keputusan K-01 dan perlakuan berita ambigu akan langsung menentukan isi B4.
4. **Penerbit berguna sebagai tipe predikat kedua** di B2–B4, sesuai kebutuhan kueri Q3 di protokol bagian 5.3.